# 02: Feature set and model family (team 04)

Builds every candidate feature, then lets refits on the data decide which stay, using splits by date.
Nothing about the result is written into this file: run it on the real data to get the real answer.

Order of the decision, from Deliverable 2: data shape, then loss, then stakeholder need, then cost,
and only then score.

> **NDA.** Aggregates only. Clear outputs before committing.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import time

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

import config as c
from features import WHY, after_shock, build_features, clean, feature_groups, split_columns, time_folds

raw, airports = c.load()
pax = clean(raw).reset_index(drop=True)
df = build_features(pax, airports)
base = f"LAG_{c.HORIZON}"
groups = feature_groups()
candidates = [base] + [col for cols in groups.values() for col in cols]
pd.set_option("display.max_colwidth", None)
print(f"{len(df):,} rows, {len(candidates)} candidate features in {len(groups)} groups, plus {base}, which always stays")

## 1. Candidate models

Both predict `log1p(target)` when the target is skewed. Trees handle gaps natively and treat airports as
categories. Ridge needs imputing, scaling and one-hot encoding. EDA section 5 covers why one ARIMA per
route is a poor fit.

In [ ]:
def log_target(model):
    return TransformedTargetRegressor(model, func=np.log1p, inverse_func=np.expm1)


def ridge(num, cat):
    lags = [col for col in num if col.startswith("LAG_")]
    other = [col for col in num if col not in lags]
    return log_target(make_pipeline(
        ColumnTransformer([
            ("lags", make_pipeline(FunctionTransformer(np.log1p), SimpleImputer(strategy="median"), StandardScaler()), lags),
            ("other", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), other),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
        ]),
        Ridge(alpha=1.0),
    ))


def trees(num, cat):
    return log_target(make_pipeline(
        ColumnTransformer([
            ("num", "passthrough", num),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), cat),
        ]),
        HistGradientBoostingRegressor(
            loss="absolute_error", random_state=0,
            categorical_features=list(range(len(num), len(num) + len(cat))),
        ),
    ))


season_lag = "LAG_12"
comparable = df[base].notna() & df[season_lag].notna()
folds = list(time_folds(df[c.DATE]))
for i, (train, test) in enumerate(folds, 1):
    print(f"fold {i}: train to {df.loc[train, c.DATE].max():%Y-%m}, test {df.loc[test, c.DATE].min():%Y-%m} to {df.loc[test, c.DATE].max():%Y-%m}")


def scores(actual, predicted):
    return {"WAPE": np.abs(actual - predicted).sum() / actual.sum(), "MdAPE": (np.abs(actual - predicted) / actual).median()}


def backtest(frame, cols, make=trees, keep=None):
    # Mean and per-fold scores. Every variant is scored on the same test rows.
    num, cat = split_columns(cols)
    rows = []
    for train, test in folds:
        test = test & comparable
        train = train if keep is None else train & keep
        model = make(num, cat).fit(frame.loc[train, num + cat], frame.loc[train, c.TARGET])
        rows.append(scores(frame.loc[test, c.TARGET], model.predict(frame.loc[test, num + cat])))
    table = pd.DataFrame(rows)
    return {"WAPE": table["WAPE"].mean(), "WAPE_std": table["WAPE"].std(), "MdAPE": table["MdAPE"].mean()}

## 2. Which months belong in training?

Months far below normal aren't like the months we'd forecast. This compares three training windows,
each scored on the same test rows, and uses the best one from here on.

- **WAPE** is total absolute error over total target, so it's weighted toward big routes.
- **MdAPE** is the median percentage error on a row, so it reflects thin routes.

In [ ]:
total = df.groupby(c.DATE)[c.TARGET].sum()
is_shock = df[c.DATE].isin(total[total < 0.7 * total.median()].index)
options = {"all months": None, "all except shock months": ~is_shock, "only after the shock": after_shock(df)}
windows = pd.DataFrame({name: backtest(df, candidates, keep=keep) for name, keep in options.items()}).T.round(4)
windows.to_csv(c.RESULTS_DIR / "training_windows.csv")
window = windows["WAPE"].idxmin()
keep = options[window]
print("Training window:", window)
windows

## 3. Does every feature group earn its place?

Permutation importance misleads when lags are this correlated, so this refits instead. Starting from every
candidate, it removes one group at a time. The group goes for good unless removing it worsens WAPE by at
least `MIN_GAIN` in `config.py`.

This table is the raw material for the per-feature rationale the brief asks for. Write that in your own
words after running on the real data.

In [ ]:
active = list(candidates)
current = backtest(df, active, keep=keep)["WAPE"]
rows = []
for group, cols in groups.items():
    without = [col for col in active if col not in cols]
    wape = backtest(df, without, keep=keep)["WAPE"]
    gain = wape - current
    stays = gain >= c.MIN_GAIN
    rows.append({"group": group, "why_test_it": WHY[group], "WAPE_without": round(wape, 4), "WAPE_cost_of_removing": round(gain, 4), "decision": "keep" if stays else "drop"})
    if not stays:
        active, current = without, wape
selection = pd.DataFrame(rows).set_index("group")
selection.to_csv(c.RESULTS_DIR / "feature_groups.csv")
kept_groups = selection.index[selection["decision"] == "keep"].tolist()
print("Final features:", active)
selection

## 4. Backtest against the baselines

In [ ]:
rows = {}
for name, lag in [("last value (naive)", base), ("same month last year (naive)", season_lag)]:
    table = pd.DataFrame([scores(df.loc[t & comparable, c.TARGET], df.loc[t & comparable, lag]) for _, t in folds])
    rows[name] = {"WAPE": table["WAPE"].mean(), "WAPE_std": table["WAPE"].std(), "MdAPE": table["MdAPE"].mean(), "fit_s": 0.0}
for name, make in [("Ridge", ridge), ("Gradient-boosted trees", trees)]:
    start = time.perf_counter()
    result = backtest(df, active, make=make, keep=keep)
    rows[name] = {**result, "fit_s": (time.perf_counter() - start) / len(folds)}
summary = pd.DataFrame(rows).T.sort_values("WAPE").round(4)
summary.to_csv(c.RESULTS_DIR / "backtest.csv")
summary

## 5. What the leak would have been worth

The same trees with revenue added. The drop in error is the illusion we avoided, and it's the number to
quote when someone asks why that column is gone.

In [ ]:
with_leak = df.join(pax[c.LEAKY])
honest = backtest(df, active, keep=keep)
leaky = backtest(with_leak, active + c.LEAKY, keep=keep)
print(f"honest:    WAPE {honest['WAPE']:.4f}, MdAPE {honest['MdAPE']:.4f}")
print(f"with leak: WAPE {leaky['WAPE']:.4f}, MdAPE {leaky['MdAPE']:.4f}")

## 6. How far ahead can this forecast?

`HORIZON` decides which lags are legal, and a PM needs to confirm the real one. Same test rows at each
horizon, against the naive forecast that's always legal up to 12 months out. If trees lose, the honest
answer at that horizon is the naive forecast, until Deliverable 4 finds something better.

In [ ]:
horizon_rows = []
naive_wape = np.mean([scores(df.loc[t & comparable, c.TARGET], df.loc[t & comparable, season_lag])["WAPE"] for _, t in folds])
for h in (1, 2, 3, 6, 12):
    frame = build_features(pax, airports, horizon=h)
    legal = feature_groups(h)
    cols = [f"LAG_{h}"] + [col for group in kept_groups if group in legal for col in legal[group]]
    result = backtest(frame, cols, keep=keep)
    horizon_rows.append({"horizon": h, "trees_WAPE": result["WAPE"], "same_month_last_year_WAPE": naive_wape, "beats_naive": result["WAPE"] < naive_wape})
horizons = pd.DataFrame(horizon_rows).set_index("horizon").round(4)
horizons.to_csv(c.RESULTS_DIR / "horizons.csv")
horizons

## 7. Decision

The answers are built from the results above when this runs, so the conclusion can't drift from the
evidence. A warning prints if the chosen model stops beating the baselines.

In [ ]:
choice = "Gradient-boosted trees"
best_baseline = summary.loc[summary.index.str.contains("naive"), "WAPE"].min()
pick = summary.loc[choice]
beats = horizons.index[horizons["beats_naive"]].tolist()
decision = pd.DataFrame([
    ("Data shape", f"{df[c.ROUTE_KEYS].drop_duplicates().shape[0]:,} routes over {df[c.DATE].nunique()} months with absent months. A single global model fits it, and trees handle the gaps and airport categories natively."),
    ("Loss", "Absolute error on log1p(target), robust to skew and outliers."),
    ("Stakeholder", f"A forecast per route per month from {len(active)} columns: {', '.join(active)}."),
    ("Cost", f"Mean fit {pick['fit_s']:.0f}s per fold, with few parameters to tune in Deliverable 4."),
    ("Score", f"WAPE {pick['WAPE']:.3f} (std {pick['WAPE_std']:.3f}) vs Ridge {summary.loc['Ridge', 'WAPE']:.3f} and best naive {best_baseline:.3f}."),
    ("Horizon limit", f"Beats same-month-last-year at horizons {beats}. Beyond that, use the naive forecast or improve the model in Deliverable 4."),
], columns=["question", "answer"])
if pick["WAPE"] >= best_baseline or pick["WAPE"] > summary.loc["Ridge", "WAPE"]:
    print("WARNING: the chosen model doesn't beat every alternative. Rethink before calling this ready.")
decision

## 8. Write FEATURES.md

The documented feature set the brief asks for, with no numbers in it, so it can be committed. It records
which data it came from: re-run on the real data before committing.

In [ ]:
lines = [
    "# Team 04 feature set",
    "",
    f"Written by `02_feature_set.ipynb` from **{'REAL' if c.DATA_DIR == c.REAL_DIR else 'FAKE'}** data.",
    "",
    f"- **Model family:** {choice}, absolute-error loss on log1p(target), forecasting {c.HORIZON} month(s) ahead.",
    f"- **Training window:** {window}.",
    f"- **Always kept:** `{base}`, the route's latest month that is known at forecast time.",
    f"- **Dropped as a leak:** {', '.join(c.LEAKY)}, known only after passengers fly.",
    f"- **How groups were chosen:** refits on date-based backtests. A group stays only if removing it worsens WAPE by at least {c.MIN_GAIN}.",
    "",
    "| Group | Columns | Decision | Why we tested it |",
    "|---|---|---|---|",
] + [f"| {g} | {', '.join(groups[g])} | {row.decision} | {row.why_test_it} |" for g, row in selection.iterrows()]
(c.ROOT / "team-04-modeling" / "FEATURES.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

In [ ]:
c.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output = c.PROCESSED_DIR / "feature_set.parquet"
rows_kept = df if keep is None else df[keep]
rows_kept[[c.DATE, c.TARGET] + active].to_parquet(output, index=False)
print(f"Saved {output.relative_to(c.ROOT)}: {len(rows_kept):,} rows, {len(active)} features")